# 03 · NLP experiment, version 2 (GPU)
Civic Operations · Group 23 · intent, slots and emergency detection

> ## For Mohammad
> Thank you for running version 1 on 26 September. This is the follow-up, and it is built to use the GPU properly.
>
> **What to do**
> 1. `git pull`, then `py -m pip install -r requirements-ml.txt`
> 2. Unzip the OneDrive data pack (`05_Data`) into the repo so you have `data/processed/`
> 3. Run this notebook top to bottom with `QUICK = False`
> 4. Send back: the executed notebook, `03_Research/experiments/results.csv`, and the `models/` folder
>
> **Expected runtime** roughly 3 to 6 hours on a 12 GB card at `QUICK = False`. Set `QUICK = True` first and
> confirm it finishes in about 10 minutes, so a typo does not cost you an afternoon.
>
> **If you run out of memory:** lower `BATCH_SIZES` to `[16]`, or drop the largest entry from `MODELS`.

## What version 1 told us, and why this version exists

| Finding from the 26 September run | What we do about it here |
|---|---|
| Validation sits at 0.97 to 0.999 while the gold set is 0.86 to 0.90, so validation cannot separate models | Report the gold set only, with bootstrap confidence intervals |
| The three learned arms are within two or three items of each other on 152 lines, so they are statistically tied | Five seeds, mean and spread, and explicit confidence intervals |
| DistilBERT had the best validation and the worst gold score: it memorised the template generator | Train on four data variants, including a noisier and a realistically-skewed one |
| `OTHER` fails on out-of-scope lines, because it has only 7 training examples | A curated out-of-scope set, plus an abstain threshold study |
| Training is near-uniform (about 518 per class) while real 311 is 64:1 imbalanced | A realistic-prior variant, class weights and focal loss |
| Latency for the embedding and transformer arms was timed on the GPU | A CPU latency benchmark for the shortlist, which is the deployment target |
| The transformer was never seeded | Every run seeded, and repeated |

In [ ]:
import sys, pathlib, json, time, random, math, itertools, csv, re
import numpy as np, pandas as pd

# ---- where are the data files? This works whether you have the whole repository
# or just the folder of files Sankalp sent you.
NEEDED = ["synthetic_nlp_utterances.parquet", "requests_real.parquet", "nlp_gold_test.csv",
          "nlp_gold_test_v2.csv", "out_of_scope_lines.csv", "intent_keywords.csv",
          "emergency_lexicon.csv"]

def find_file(name):
    here = pathlib.Path.cwd().resolve()
    roots = [here, *here.parents][:4]
    places = []
    for r in roots:
        places += [r, r / "data", r / "data" / "processed", r / "data" / "reference",
               r / "01_Application" / "data" / "processed", r / "01_Application" / "data" / "reference"]
    for d in places:
        if (d / name).exists():
            return d / name
    raise FileNotFoundError(
        "Could not find " + name + ". Put all of these next to this notebook, or in a "
        "'data' folder beside it: " + ", ".join(NEEDED) +
        ".  I looked in: " + " | ".join(str(d) for d in places[:8]))

FILES = {n: find_file(n) for n in NEEDED}
ROOT = pathlib.Path.cwd().resolve()
EXP = ROOT / "experiments"; EXP.mkdir(exist_ok=True)
MODELS_DIR = ROOT / "models"; MODELS_DIR.mkdir(exist_ok=True)
print("data files found:")
for n, f in FILES.items():
    print(f"   {n:<38} {f.parent}")


### Pre-flight

Run this before anything else. It checks the packages, the files and the GPU, and tells you plainly what is
missing. If this cell is happy, nothing later should fail for environment reasons.

In [ ]:
import importlib
problems = []
for mod, why in [("torch", "training"), ("transformers", "the transformer arms"),
                 ("sentence_transformers", "the MiniLM arm"), ("sklearn", "TF-IDF and metrics"),
                 ("matplotlib", "the plots"), ("joblib", "saving models")]:
    try:
        m = importlib.import_module(mod)
        print(f"  ok   {mod:<24} {getattr(m, '__version__', '')}")
    except ImportError:
        problems.append(f"{mod} is missing ({why}). Install with: py -m pip install {mod}")

try:
    importlib.import_module("sentencepiece")
    print("  ok   sentencepiece            (needed by DeBERTa)")
except ImportError:
    print("  note sentencepiece missing: DeBERTa will be skipped automatically, the run still completes.")

import torch
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, brier_score_loss
import matplotlib
if "ipykernel" not in sys.modules:      # executed as a plain script: never block on a plot window
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

QUICK = True            # <-- set to False for the real run
SEEDS = [2026, 7, 13] if QUICK else [2026, 7, 13, 42, 99]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP = DEVICE == "cuda"          # half precision: about twice as fast, and halves memory
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == "cuda" else 0

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def free():
    """Release GPU memory between arms. Safe to call on a CPU-only machine."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if problems:
    print("")
    print("STOP, fix these first:")
    for problem in problems:
        print("  -", problem)
else:
    print("")
    print(f"device: {DEVICE} | quick: {QUICK} | mixed precision: {AMP}")
    if DEVICE == "cuda":
        print("gpu:", torch.cuda.get_device_name(0), "| memory: %.1f GB" % VRAM_GB)
        if VRAM_GB < 6:
            print("  Small card. Batch sizes drop automatically if it runs out of memory,")
            print("  and DeBERTa is skipped if it will not fit. Expect 6 to 9 hours for the full run.")
        else:
            print("  Expect roughly 3 to 6 hours for the full run.")
    else:
        print("No GPU visible. The notebook still runs, but set QUICK = True or it will take many hours.")


## A · Data

Four training variants. They all use the same hand-written gold set for scoring, which is the only honest test we
have: the real 311 "text" is 253 fixed dropdown strings chosen by an agent, not citizen language, so it cannot be
used to train or test a bot that reads what a person wrote.

In [ ]:
utt  = pd.read_parquet(FILES["synthetic_nlp_utterances.parquet"])
gold_v1 = pd.read_csv(FILES["nlp_gold_test.csv"]).assign(source="v1 team-written")
gold_v2 = pd.read_csv(FILES["nlp_gold_test_v2.csv"]).assign(source="v2 written 5 Oct")
gold = pd.concat([gold_v1, gold_v2], ignore_index=True)
GOLD_SRC = gold.source.to_numpy()
oos  = pd.read_csv(FILES["out_of_scope_lines.csv"])
real = pd.read_parquet(FILES["requests_real.parquet"], columns=["request_type_code", "borough"])

train0, val = utt[utt.split == "train"].copy(), utt[utt.split == "val"].copy()
LABELS = sorted(utt.intent.unique())

# --- leakage: the gold set is our only unseen test, so nothing in it may be trained on
leaked = train0.text.isin(set(gold.text))
print("removed", int(leaked.sum()), "training rows that appear word for word in the gold set:",
      list(train0.loc[leaked, "text"])[:5])
train0 = train0[~leaked].copy()

# --- the type distribution, corrected for how we sampled
# We took about 10,000 rows per borough, but the city does not report that way: Brooklyn files
# about 1.21M requests a year and Staten Island about 160k. Taking the raw share from our sample
# over-weights Staten Island roughly sixfold. Post-stratifying fixes it with the data we already have.
TRUE_BOROUGH_SHARE = {"BROOKLYN": .3055, "QUEENS": .2442, "BRONX": .2154,
                      "MANHATTAN": .1943, "STATEN ISLAND": .0406}
within = real.groupby("borough").request_type_code.value_counts(normalize=True).unstack(fill_value=0)
real_share = sum(within.loc[b] * wgt for b, wgt in TRUE_BOROUGH_SHARE.items() if b in within.index)
real_share = (real_share / real_share.sum()).sort_values(ascending=False)

naive = real.request_type_code.value_counts(normalize=True)
shift = (real_share - naive.reindex(real_share.index).fillna(0)).abs().sum() * 100
print(f"type distribution corrected for borough sampling: {shift:.1f} percentage points of total shift")

print(f"{len(train0):,} train | {len(val):,} val | {len(gold)} gold ({len(gold_v1)} v1 + {len(gold_v2)} v2) | {len(oos)} out-of-scope lines")
print(f"{len(LABELS)} intents")
print("\ntraining distribution now  : min %d, max %d per intent" %
      (train0.intent.value_counts().min(), train0.intent.value_counts().max()))
print("real 311 distribution      : min %.4f, max %.4f share  (%.0f:1)" %
      (real_share.min(), real_share.max(), real_share.max() / real_share.min()))
print("gold lines per intent      : min %d, max %d" %
      (gold.intent.value_counts().min(), gold.intent.value_counts().max()))

### A1 · Augmentation, so training text looks as messy as what people actually type

In [ ]:
ASR = {"there is": "theres", "it is": "its", "i am": "im", "cannot": "cant", "street": "st",
       "avenue": "ave", "building": "bldg", "apartment": "apt", "because": "bc", "please": "pls"}

def noisy(text, rng):
    """One realistic corruption: a typo, a contraction, lost punctuation, shouting, or truncation."""
    kind = rng.integers(0, 5)
    if kind == 0 and len(text) > 12:                      # transposed or dropped character
        i = rng.integers(1, len(text) - 1)
        return text[:i] + text[i + 1:] if rng.random() < .5 else text[:i] + text[i + 1] + text[i] + text[i + 2:]
    if kind == 1:
        for a, b in ASR.items():
            text = text.replace(a, b)
        return text
    if kind == 2:
        return re.sub(r"[.,!?']", "", text)
    if kind == 3:
        return text.upper() if rng.random() < .5 else text.lower()
    words = text.split()
    return " ".join(words[:max(3, len(words) - rng.integers(1, 4))])  # cut off mid-sentence

def augment_df(df, frac=0.5, seed=0):
    rng = np.random.default_rng(seed)
    extra = df.sample(frac=frac, random_state=seed).copy()
    extra["text"] = [noisy(t, rng) for t in extra.text]
    extra["has_slot_labels"] = False                      # the spans no longer line up after corruption
    return pd.concat([df, extra], ignore_index=True)

def with_out_of_scope(df, copies=8, seed=0):
    """OTHER has 7 training examples today, which is why out-of-scope lines misroute."""
    rng = np.random.default_rng(seed)
    rows = []
    for _ in range(copies):
        for t, intent in zip(oos.text, oos.intent):
            rows.append(dict(text=noisy(t, rng) if rng.random() < .4 else t, intent=intent,
                             emergency_category="NONE", profanity="NONE", has_slot_labels=False,
                             split="train", is_synthetic=True))
    return pd.concat([df, pd.DataFrame(rows)], ignore_index=True)

def realistic_prior(df, floor=40, seed=0):
    """Resample so the request types appear as often as they do in real NYC 311.
    EMERGENCY, OTHER and SMALLTALK are not request types, so they keep a fixed share."""
    rng = np.random.default_rng(seed)
    n_total = len(df)
    keep = []
    for intent, group in df.groupby("intent"):
        if intent in ("EMERGENCY",):
            target = int(0.04 * n_total)
        elif intent in ("OTHER", "SMALLTALK"):
            target = int(0.03 * n_total)
        else:
            target = int(real_share.get(intent, 1 / len(LABELS)) * 0.93 * n_total)
        target = max(floor, target)
        keep.append(group.sample(target, replace=target > len(group), random_state=int(rng.integers(1e6))))
    return pd.concat(keep, ignore_index=True)

In [ ]:
VARIANTS = {
    "D0 uniform":            with_out_of_scope(train0, copies=0),
    "D1 realistic":          realistic_prior(with_out_of_scope(train0, copies=4)),
    "D2 uniform+noise":      augment_df(with_out_of_scope(train0, copies=4)),
    "D3 realistic+noise":    augment_df(realistic_prior(with_out_of_scope(train0, copies=6))),
}
if QUICK:   # a smaller copy of each variant, so a typo costs minutes instead of hours.
    # NOTE: sample at random rather than capping per intent. Capping per intent would flatten
    # D1 and D3 back towards uniform and the comparison would be meaningless.
    VARIANTS = {k: v.sample(n=min(len(v), 3000), random_state=1) for k, v in VARIANTS.items()}

summary = pd.DataFrame({k: dict(rows=len(v), intents=v.intent.nunique(),
                                min_per_intent=v.intent.value_counts().min(),
                                max_per_intent=v.intent.value_counts().max(),
                                other_lines=int((v.intent == "OTHER").sum()))
                        for k, v in VARIANTS.items()}).T
summary

## B · Scoring harness

Everything is scored on the gold set, every number carries a bootstrap confidence interval, and every run appends a
row to `03_Research/experiments/results.csv`. Nothing goes into the report that is not a row in that file.

In [ ]:
RESULTS_PATH = EXP / "results.csv"
FIELDS = ["run_at", "task", "arm", "variant", "config", "seed", "split", "accuracy", "macro_f1",
          "macro_f1_lo", "macro_f1_hi", "macro_f1_v1", "macro_f1_v2", "ms_per_item", "size_mb", "notes"]

SESSION_ROWS = []           # this run only, so a leftover file cannot confuse the decision below

if RESULTS_PATH.exists():
    with open(RESULTS_PATH, newline="", encoding="utf-8") as _f:
        _head = next(csv.reader(_f), [])
    if _head != FIELDS:
        _backup = RESULTS_PATH.with_name(f"results_before_{pd.Timestamp.now():%Y%m%d_%H%M%S}.csv")
        RESULTS_PATH.rename(_backup)
        print(f"an older results.csv was here with different columns; kept as {_backup.name}")

def log_result(**kw):
    row = {f: kw.get(f, "") for f in FIELDS}
    row["run_at"] = pd.Timestamp.now().isoformat(timespec="seconds")
    new = not RESULTS_PATH.exists()
    with open(RESULTS_PATH, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new:
            w.writeheader()
        w.writerow(row)
    SESSION_ROWS.append(row)
    return row

def bootstrap_ci(y_true, y_pred, n=600, seed=0):
    """95% interval for macro-F1. With 152 gold lines this is wide, and saying so is the point."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y_true))
    vals = [f1_score(y_true[s], y_pred[s], average="macro", zero_division=0)
            for s in (rng.choice(idx, len(idx), replace=True) for _ in range(n))]
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def score(task, arm, variant, config, seed, y_true, y_pred, ms=0.0, size_mb=0.0, notes=""):
    lo, hi = bootstrap_ci(y_true, y_pred, seed=seed)
    yt, yp = np.asarray(y_true), np.asarray(y_pred)
    # the two slices test different things: v1 uses the obvious wording, v2 deliberately avoids it.
    # A large gap between them is informative, so both are recorded on every row.
    def slice_f1(tag):
        m = GOLD_SRC == tag
        return round(f1_score(yt[m], yp[m], average="macro", zero_division=0), 4) if m.any() else ""
    row = log_result(task=task, arm=arm, variant=variant, config=json.dumps(config), seed=seed, split="gold",
                     accuracy=round(accuracy_score(yt, yp), 4),
                     macro_f1=round(f1_score(yt, yp, average="macro", zero_division=0), 4),
                     macro_f1_lo=round(lo, 4), macro_f1_hi=round(hi, 4),
                     macro_f1_v1=slice_f1("v1 team-written"), macro_f1_v2=slice_f1("v2 written 5 Oct"),
                     ms_per_item=round(ms, 3), size_mb=round(size_mb, 2), notes=notes)
    print(f"{arm:<22} {variant:<18} macro-F1 {row['macro_f1']:.3f}  [{lo:.3f}, {hi:.3f}]"
          f"   v1 {row['macro_f1_v1']}  v2 {row['macro_f1_v2']}")
    return row

## C · Intent, stage 1: every arm on every data variant

Defaults only at this stage. The question here is **which training data wins**, not which hyperparameters.

In [ ]:
# --- A0 keyword baseline (self-contained, no package needed)
KEYWORDS = {r["request_type_code"]: r["keywords"] for r in csv.DictReader(open(FILES["intent_keywords.csv"], encoding="utf-8"))}
EM_PAT = [re.compile(r["pattern"], re.I) for r in csv.DictReader(open(FILES["emergency_lexicon.csv"], encoding="utf-8"))]

def a0_predict(texts):
    out = []
    for t in texts:
        if any(p.search(t) for p in EM_PAT):
            out.append("EMERGENCY"); continue
        words = set(re.findall(r"[a-z\-]+", t.lower()))
        sc = {c: len(words & set(k.split())) for c, k in KEYWORDS.items()}
        best = max(sc, key=sc.get)
        out.append(best if sc[best] else "OTHER")
    return out

t0 = time.perf_counter()
pred = a0_predict(gold.text.tolist())
ms = (time.perf_counter() - t0) * 1000 / len(gold)
score("intent", "A0 keywords", "n/a", {}, 0, gold.intent, pred, ms, 0.01, "rules, unchanged from v1")

In [ ]:
from sklearn.pipeline import make_pipeline, make_union
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
import pickle

def fit_tfidf(df, calibrated=False, seed=2026):
    feats = make_union(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2),
                       TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True, min_df=2))
    clf = LinearSVC(C=0.8, class_weight="balanced")
    model = make_pipeline(feats, CalibratedClassifierCV(clf, cv=3) if calibrated else clf)
    return model.fit(df.text, df.intent)

for name, df in VARIANTS.items():
    for seed in SEEDS[:1]:
        set_seed(seed)
        m = fit_tfidf(df, seed=seed)
        t0 = time.perf_counter(); pred = m.predict(gold.text); ms = (time.perf_counter() - t0) * 1000 / len(gold)
        score("intent", "A1 tfidf+svm", name, dict(C=0.8, class_weight="balanced"), seed,
              gold.intent, pred, ms, len(pickle.dumps(m)) / 1e6)

In [ ]:
from sentence_transformers import SentenceTransformer

enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
def embed(texts, bs=256):
    return enc.encode(list(texts), batch_size=bs, normalize_embeddings=True, show_progress_bar=False)

gold_emb = embed(gold.text)
for name, df in VARIANTS.items():
    set_seed(SEEDS[0])
    X = embed(df.text)
    clf = LogisticRegression(max_iter=3000, C=4.0, class_weight="balanced").fit(X, df.intent)
    t0 = time.perf_counter(); pred = clf.predict(gold_emb); ms = (time.perf_counter() - t0) * 1000 / len(gold)
    score("intent", "A2 minilm+logreg", name, dict(C=4.0, class_weight="balanced"), SEEDS[0],
          gold.intent, pred, ms, 90.9)

### C1 · Transformer arms

Three families, so we can say something about size rather than guessing: DistilBERT (66M), RoBERTa base (125M) and
DeBERTa v3 small (142M, but a stronger encoder). Each can be trained with plain cross-entropy, class weights, or
focal loss, which is the standard answer to heavy imbalance.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader

WANTED = ["distilbert-base-uncased"] if QUICK else [
    "distilbert-base-uncased", "roberta-base", "microsoft/deberta-v3-small",
]
MODELS = []
for name in WANTED:                       # drop anything this machine cannot actually load
    try:
        AutoTokenizer.from_pretrained(name)
        MODELS.append(name)
    except Exception as e:
        print(f"skipping {name}: {type(e).__name__} - {str(e)[:90]}")
print("models in this run:", MODELS)
L2I = {l: i for i, l in enumerate(LABELS)}

class FocalLoss(torch.nn.Module):
    """Down-weights the easy, frequent classes so the rare ones still produce gradient."""
    def __init__(self, gamma=2.0, weight=None):
        super().__init__(); self.gamma, self.weight = gamma, weight
    def forward(self, logits, target):
        ce = torch.nn.functional.cross_entropy(logits, target, weight=self.weight, reduction="none")
        return ((1 - torch.exp(-ce)) ** self.gamma * ce).mean()

def _train_once(model_name, df, lr, epochs, batch, max_len, warmup, weight_decay,
                label_smoothing, loss, focal_gamma, seed):
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(LABELS)).to(DEVICE)
    ys = df.intent.map(L2I).to_numpy()
    counts = np.bincount(ys, minlength=len(LABELS)).astype(float)
    w = torch.tensor((counts.sum() / np.maximum(counts, 1)) / len(LABELS), dtype=torch.float, device=DEVICE)
    if loss == "ce":
        crit = torch.nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    elif loss == "weighted":
        crit = torch.nn.CrossEntropyLoss(weight=w, label_smoothing=label_smoothing)
    else:
        crit = FocalLoss(gamma=focal_gamma, weight=w)
    loader = DataLoader(list(zip(df.text.tolist(), ys)), batch_size=batch, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    steps = max(1, len(loader) * epochs)
    sched = get_linear_schedule_with_warmup(opt, int(warmup * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    model.train()
    for _ in range(epochs):
        for texts, yb in loader:
            b = tok(list(texts), padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(DEVICE)
            with torch.amp.autocast("cuda", enabled=AMP):
                l = crit(model(**b).logits, yb.to(DEVICE))
            scaler.scale(l).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()
    model.eval()
    return model, tok


def train_transformer(model_name, df, lr=3e-5, epochs=3, batch=32, max_len=64, warmup=0.1,
                      weight_decay=0.01, label_smoothing=0.0, loss="ce", focal_gamma=2.0, seed=2026):
    """Trains, and if the card runs out of memory, halves the batch and tries again.
    A 4 GB laptop card and a 24 GB desktop card both finish; only the batch size differs."""
    for attempt in range(3):
        try:
            return _train_once(model_name, df, lr, epochs, batch, max_len, warmup,
                               weight_decay, label_smoothing, loss, focal_gamma, seed)
        except torch.cuda.OutOfMemoryError:
            free()
            batch = max(4, batch // 2)
            print(f"   out of memory, retrying {model_name} at batch {batch}")
    raise RuntimeError(f"{model_name} does not fit on this card even at batch 4")

@torch.no_grad()
def transformer_logits(model, tok, texts, max_len=64, batch=64):
    out = []
    for i in range(0, len(texts), batch):
        b = tok(list(texts[i:i + batch]), padding=True, truncation=True, max_length=max_len,
                return_tensors="pt").to(DEVICE)
        out.append(model(**b).logits.float().cpu())
    return torch.cat(out)

def transformer_predict(model, tok, texts, **kw):
    return [LABELS[i] for i in transformer_logits(model, tok, list(texts), **kw).argmax(-1).tolist()]

In [ ]:
gold_texts = gold.text.tolist()
for model_name in MODELS:
    for vname, df in VARIANTS.items():
        m, tk = train_transformer(model_name, df, seed=SEEDS[0])
        t0 = time.perf_counter(); pred = transformer_predict(m, tk, gold_texts)
        ms = (time.perf_counter() - t0) * 1000 / len(gold_texts)
        size = sum(p.numel() for p in m.parameters()) * 4 / 1e6
        score("intent", f"A3 {model_name.split('/')[-1]}", vname,
              dict(lr=3e-5, epochs=3, batch=32, loss="ce"), SEEDS[0], gold.intent, pred, ms, size)
        del m; free()

## D · Intent, stage 2: hyperparameter search

A random search over the grid below, on the winning data variant from stage 1. Random search rather than a full grid
because it covers more of each dimension for the same budget.

In [ ]:
res = pd.DataFrame(SESSION_ROWS)
ranked = res[(res.task == "intent") & (res.variant != "n/a")].sort_values("macro_f1", ascending=False)
best_variant = ranked.iloc[0]["variant"]
best_family = ranked.iloc[0]["arm"]
print("stage 1 winner:", best_family, "on", best_variant)

GRID = dict(lr=[1e-5, 2e-5, 3e-5, 5e-5], epochs=[2, 3, 4, 6], batch=[16, 32],
            max_len=[48, 64, 96], warmup=[0.0, 0.1], weight_decay=[0.0, 0.01, 0.1],
            label_smoothing=[0.0, 0.1], loss=["ce", "weighted", "focal"])
BUDGET = 2 if QUICK else 18
SWEEP_MODEL = "microsoft/deberta-v3-small" if (not QUICK and "microsoft/deberta-v3-small" in MODELS) else MODELS[0]

rng = np.random.default_rng(2026)
tried = []
for trial in range(BUDGET):
    cfg = {k: v[int(rng.integers(len(v)))] for k, v in GRID.items()}
    m, tk = train_transformer(SWEEP_MODEL, VARIANTS[best_variant], seed=SEEDS[0], **cfg)
    pred = transformer_predict(m, tk, gold_texts, max_len=cfg["max_len"])
    row = score("intent", f"A4 sweep {SWEEP_MODEL.split('/')[-1]}", best_variant, cfg, SEEDS[0],
                gold.intent, pred, notes=f"trial {trial + 1}/{BUDGET}")
    tried.append((row["macro_f1"], cfg))
    del m; free()

best_f1, best_cfg = max(tried, key=lambda x: x[0])
print("\nbest configuration:", best_cfg, "->", best_f1)

## E · Intent, stage 3: the chosen configuration, repeated over seeds

One run proves nothing. This is where the number for the report comes from: mean and spread across seeds.

In [ ]:
runs = []
for seed in SEEDS:
    m, tk = train_transformer(SWEEP_MODEL, VARIANTS[best_variant], seed=seed, **best_cfg)
    pred = transformer_predict(m, tk, gold_texts, max_len=best_cfg["max_len"])
    runs.append(score("intent", f"A5 final {SWEEP_MODEL.split('/')[-1]}", best_variant, best_cfg, seed,
                      gold.intent, pred, notes="final, seeded")["macro_f1"])
    if seed != SEEDS[-1]:
        del m; free()

print("\nmacro-F1 over %d seeds: mean %.3f, sd %.3f, range %.3f to %.3f" %
      (len(runs), np.mean(runs), np.std(runs), min(runs), max(runs)))
final_model, final_tok = m, tk

## F · Calibration and the abstain threshold

Two separate questions. Is the probability honest (calibration)? And at what confidence should the bot say
*“I am not sure, a person will read this”* rather than guess a request type (abstain)?

In [ ]:
probs = torch.softmax(transformer_logits(final_model, final_tok, gold_texts,
                                         max_len=best_cfg["max_len"]), dim=-1).numpy()
pred_idx = probs.argmax(1); conf = probs.max(1)
correct = np.array([LABELS[i] for i in pred_idx]) == gold.intent.to_numpy()

bins = np.linspace(0, 1, 11)
which = np.digitize(conf, bins) - 1
ece = sum(abs(correct[which == b].mean() - conf[which == b].mean()) * (which == b).sum() / len(conf)
          for b in range(10) if (which == b).sum())
print("expected calibration error: %.3f" % ece)
print("Brier (one-vs-rest on the predicted class): %.3f" % brier_score_loss(correct, conf))

rows = []
for t in np.arange(0.2, 0.96, 0.05):
    abstained = conf < t
    kept = ~abstained
    rows.append(dict(threshold=round(t, 2), abstain_rate=round(abstained.mean(), 3),
                     accuracy_on_kept=round(correct[kept].mean(), 3) if kept.any() else np.nan,
                     out_of_scope_caught=round(abstained[gold.intent.isin(["OTHER", "SMALLTALK"]).to_numpy()].mean(), 3)))
abstain = pd.DataFrame(rows)
display(abstain)
plt.figure(figsize=(6, 3))
plt.plot(abstain.threshold, abstain.accuracy_on_kept, marker="o", label="accuracy on kept")
plt.plot(abstain.threshold, abstain.abstain_rate, marker="s", label="abstain rate")
plt.xlabel("confidence threshold"); plt.legend(); plt.grid(alpha=.3); plt.title("Abstain study"); plt.show()

## G · CPU latency for the shortlist

The deployment argument is a free CPU host, so GPU timings prove nothing. This measures what the citizen waits.

In [ ]:
cpu_rows = []
sample = (gold.text.tolist() * 4)[:200]

tfidf = fit_tfidf(VARIANTS[best_variant])
t0 = time.perf_counter(); tfidf.predict(sample)
cpu_rows.append(dict(model="TF-IDF + SVM", ms_per_item=(time.perf_counter() - t0) * 1000 / len(sample),
                     size_mb=round(len(pickle.dumps(tfidf)) / 1e6, 1)))

torch.set_num_threads(4)
cpu_model = final_model.to("cpu").eval()
with torch.no_grad():
    t0 = time.perf_counter()
    for i in range(0, len(sample), 16):
        b = final_tok(sample[i:i + 16], padding=True, truncation=True,
                      max_length=best_cfg["max_len"], return_tensors="pt")
        cpu_model(**b)
cpu_rows.append(dict(model=SWEEP_MODEL, ms_per_item=(time.perf_counter() - t0) * 1000 / len(sample),
                     size_mb=round(sum(p.numel() for p in cpu_model.parameters()) * 4 / 1e6, 1)))
final_model.to(DEVICE)
pd.DataFrame(cpu_rows).round(2)

## H · Emergency detection

Recall first: a missed emergency is the worst failure in the system. Version 1 reached 0.947 with one medical case
missed. This sweeps the threshold properly, reports recall per category, checks a named false-alarm set, and tests
the proposed immobility patch.

In [ ]:
NEGATION = re.compile(r"\b(no one|nobody|not) (is )?(hurt|injured|inside|stuck|trapped)\b", re.I)
LEX = [(re.compile(r["pattern"], re.I), float(r["weight"])) for r in
       csv.DictReader(open(FILES["emergency_lexicon.csv"], encoding="utf-8"))]
# The patch proposed after v1: the one gold miss was "the driver is not moving".
IMMOBILITY = [(re.compile(p, re.I), 0.9) for p in
              [r"\bnot moving\b", r"\bunconscious\b", r"\bnot breathing\b", r"\btrapped\b",
               r"\bunresponsive\b", r"\bpinned under\b"]]

def rule_score(text, lexicon):
    hits = [w for pat, w in lexicon if pat.search(text)]
    if not hits:
        return 0.0
    return max(hits) * (0.6 if NEGATION.search(text) else 1.0)

is_em = (gold.intent == "EMERGENCY").to_numpy()
FALSE_ALARM = gold.text.str.contains("hydrant|fire escape|fire lane", case=False).to_numpy()

em_clf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True),
                       LogisticRegression(max_iter=2000, class_weight="balanced"))
em_clf.fit(VARIANTS[best_variant].text, VARIANTS[best_variant].intent == "EMERGENCY")
proba = em_clf.predict_proba(gold.text)[:, 1]

rows = []
for label, lexicon in [("rules v1", LEX), ("rules + immobility patch", LEX + IMMOBILITY)]:
    rules = np.array([rule_score(t, lexicon) > 0 for t in gold.text])
    for thr in [None, 0.5, 0.45, 0.4, 0.35, 0.3, 0.25]:
        pred = rules if thr is None else (rules | (proba >= thr))
        p, r, f, _ = precision_recall_fscore_support(is_em, pred, average="binary", zero_division=0)
        rows.append(dict(lexicon=label, model_threshold="rules only" if thr is None else thr,
                         recall=round(r, 3), precision=round(p, 3), f1=round(f, 3),
                         missed=int((is_em & ~pred).sum()), false_alarms=int((~is_em & pred).sum()),
                         false_alarms_named_set=int((pred & FALSE_ALARM & ~is_em).sum())))
em_table = pd.DataFrame(rows)
display(em_table)

best_em = em_table[em_table.recall >= 0.999].sort_values("precision", ascending=False)
print("\nconfigurations reaching recall 1.0:", len(best_em))
display(best_em.head(3))

In [ ]:
# which emergencies are still missed, and in which category
rules_patched = np.array([rule_score(t, LEX + IMMOBILITY) > 0 for t in gold.text])
pred = rules_patched | (proba >= 0.35)
missed = gold[is_em & ~pred]
print("still missed:", len(missed))
display(missed[["text", "emergency_category"]])

by_cat = (gold[is_em].assign(caught=pred[is_em]).groupby("emergency_category")
          .agg(n=("text", "size"), recall=("caught", "mean")).round(3))
display(by_cat)

## I · Slots

The 0.965 span F1 from version 1 was measured on template text. Until hand-labelled spans exist, this re-measures it
on the noisy variant, which is a harder and fairer proxy.

In [ ]:
from transformers import AutoModelForTokenClassification

def spans(tags):
    out, cur = set(), None
    for i, t in enumerate(tags):
        if t.startswith("B-"):
            cur = [t[2:], i, i]; out.add(tuple(cur))
        elif t.startswith("I-") and cur and t[2:] == cur[0]:
            out.discard(tuple(cur)); cur[2] = i; out.add(tuple(cur))
        else:
            cur = None
    return out

def span_f1(gold_sets, pred_sets):
    tp = sum(len(g & p) for g, p in zip(gold_sets, pred_sets))
    fp = sum(len(p - g) for g, p in zip(gold_sets, pred_sets))
    fn = sum(len(g - p) for g, p in zip(gold_sets, pred_sets))
    pr, rc = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return dict(precision=round(pr, 3), recall=round(rc, 3), f1=round(2 * pr * rc / max(pr + rc, 1e-9), 3))

slot_train = train0[train0.has_slot_labels]
slot_eval = val[val.has_slot_labels].head(400 if QUICK else 3000)
TAGS = sorted({t for ts in utt[utt.has_slot_labels].bio_tags for t in ts}); T2I = {t: i for i, t in enumerate(TAGS)}
print(len(slot_train), "slot-labelled training rows |", len(TAGS), "tags")

for slot_lr, slot_epochs in ([(5e-5, 1)] if QUICK else [(3e-5, 3), (5e-5, 5)]):
    set_seed(SEEDS[0])
    tok = AutoTokenizer.from_pretrained(MODELS[0])
    mdl = AutoModelForTokenClassification.from_pretrained(MODELS[0], num_labels=len(TAGS)).to(DEVICE)
    opt = torch.optim.AdamW(mdl.parameters(), lr=slot_lr)

    def encode(tokens_list, tags_list=None):
        enc_ = tok([list(t) for t in tokens_list], is_split_into_words=True, truncation=True,
                   max_length=64, padding=True, return_tensors="pt")
        if tags_list is None:
            return enc_
        labels = []
        for i, tags in enumerate(tags_list):
            wid, prev, lab = enc_.word_ids(i), None, []
            for w in wid:
                lab.append(-100 if w is None or w == prev else T2I[tags[w]]); prev = w
            labels.append(lab)
        enc_["labels"] = torch.tensor(labels)
        return enc_

    mdl.train()
    for _ in range(slot_epochs):
        for i in range(0, len(slot_train), 32):
            chunk = slot_train.iloc[i:i + 32]
            b = encode(chunk.tokens, chunk.bio_tags).to(DEVICE)
            mdl(**b).loss.backward(); opt.step(); opt.zero_grad()
    mdl.eval()

    @torch.no_grad()
    def tag_one(tokens):
        e = encode([tokens]).to(DEVICE)
        p = mdl(**e).logits.argmax(-1)[0].tolist()
        out, prev = ["O"] * len(tokens), None
        for k, w in enumerate(e.word_ids(0)):
            if w is not None and w != prev:
                out[w] = TAGS[p[k]]
            prev = w
        return out

    gs = [spans(list(g)) for g in slot_eval.bio_tags]
    ps = [spans(tag_one(list(t))) for t in slot_eval.tokens]
    sc = span_f1(gs, ps)
    log_result(task="slots", arm=MODELS[0], variant="val template", seed=SEEDS[0],
               config=json.dumps(dict(lr=slot_lr, epochs=slot_epochs)), split="val",
               macro_f1=sc["f1"], notes=f"precision {sc['precision']}, recall {sc['recall']}")
    print(f"lr={slot_lr} epochs={slot_epochs} -> {sc}")
    del mdl; free()

## J · Results, decision and artifacts

In [ ]:
res = pd.DataFrame(SESSION_ROWS)
intent = res[res.task == "intent"].copy()
for col in ["macro_f1", "macro_f1_lo", "macro_f1_hi", "macro_f1_v1", "macro_f1_v2", "ms_per_item", "size_mb"]:
    intent[col] = pd.to_numeric(intent[col], errors="coerce")
table = (intent.groupby(["arm", "variant"])
         .agg(macro_f1=("macro_f1", "mean"), sd=("macro_f1", "std"),
              lo=("macro_f1_lo", "mean"), hi=("macro_f1_hi", "mean"),
              ms=("ms_per_item", "mean"), size_mb=("size_mb", "mean"), runs=("macro_f1", "size"))
         .sort_values("macro_f1", ascending=False).round(3))
display(table.head(20))

print("")
print("The two gold slices, for the chosen arm:")
print("  v1 was written by the team with the natural wording for each type.")
print("  v2 was written on 5 October and deliberately avoids the obvious keywords.")
print("  A model that scores well on v1 and badly on v2 has learned vocabulary, not meaning.")
display(intent.groupby("arm")[["macro_f1_v1", "macro_f1_v2"]].mean().round(3).sort_values("macro_f1_v2", ascending=False))

print("\nDecision rule, agreed before the run:")
print("  choose the smallest model whose mean gold macro-F1 is within 2 points of the best,")
print("  provided CPU latency is under 20 ms per utterance and emergency recall is 1.0.")
top = table.macro_f1.max()
eligible = table[(table.macro_f1 >= top - 0.02)].sort_values("size_mb")
print("\nEligible, smallest first:")
display(eligible)

In [ ]:
import joblib, hashlib, datetime
chosen_tfidf = fit_tfidf(VARIANTS[best_variant])
joblib.dump(chosen_tfidf, MODELS_DIR / "c01_intent_tfidf_svm.joblib")
joblib.dump(em_clf, MODELS_DIR / "c02_emergency_tfidf_lr.joblib")
final_model.save_pretrained(MODELS_DIR / "c01_intent_transformer")
final_tok.save_pretrained(MODELS_DIR / "c01_intent_transformer")

man = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()[:16] for p in MODELS_DIR.glob("*.joblib")}
man.update(_trained_at=datetime.datetime.now().isoformat(timespec="seconds"), _quick=QUICK,
           _device=DEVICE, _seeds=SEEDS, _best_variant=best_variant, _best_config=best_cfg,
           _emergency_threshold=0.35, _emergency_lexicon="emergency_lexicon.csv + immobility patch",
           _abstain_threshold="see section F")
(MODELS_DIR / "manifest.json").write_text(json.dumps(man, indent=2))
print(json.dumps(man, indent=2))
print("\nSend back: this notebook, 03_Research/experiments/results.csv, and the models/ folder.")

## What I will do with the results

1. Take the chosen configuration into the intake component, keeping the keyword rules as the fallback.
2. Fix the emergency lexicon at whichever configuration reaches **recall 1.0** with the fewest false alarms, and
   record the threshold in the manifest, because the model file alone does not reproduce the behaviour.
3. Set the abstain threshold from section F, so out-of-scope messages go to a person instead of being guessed.
4. Report the CPU latency, not the GPU latency, as the number that supports a free-host deployment.
5. Put the mean and spread over seeds in the report, with the confidence interval, and say plainly when arms are
   statistically tied.